In [1]:
import pandas as pd

from src.data_paths import (
    FOLD_TRIAL_IDS_FILENAME_TEMPLATE,
    GATHERERS_FOLDS_DIR,
    GATHERERS_PROCESSED_PATH,
    HUNTERS_FOLDS_DIR,
    HUNTERS_PROCESSED_PATH,
    HUNTING_IS_CORRECT_ALL_FOLDS_PATH,
)

In [2]:
# ---------------------------
# paths
# ---------------------------
output_dir = HUNTERS_FOLDS_DIR
output_dir.mkdir(parents=True, exist_ok=True)

# ---------------------------
# load
# ---------------------------
hunters = pd.read_csv(HUNTERS_PROCESSED_PATH)
hunt_all_folds = pd.read_csv(HUNTING_IS_CORRECT_ALL_FOLDS_PATH)

In [3]:

# ---------------------------
# build gatherer-like regime column
# ---------------------------
folds_df = hunt_all_folds.copy()

regime_map = {
    "train": "train",
    "new_item": "seen_subject_unseen_item",
    "new_subject": "unseen_subject_seen_item",
    "new_item_and_subject": "unseen_subject_unseen_item",
}

folds_df["regime"] = (
    folds_df["eval_type"].astype(str)
    + "_"
    + folds_df["eval_regime"].map(regime_map)
)

# ---------------------------
# join metadata from hunters
# ---------------------------
# keep only columns we actually want to bring in
hunters_join_cols = [
    "participant_id",
    "article_batch",
    "article_id",
    "text_id",                  # becomes unique_paragraph_id
    "list_number",
    "question_preview",
    "repeated_reading_trial",
    "is_correct",
]

hunters_meta = hunters[hunters_join_cols].drop_duplicates()

merged = folds_df.merge(
    hunters_meta,
    on=["participant_id", "article_batch", "article_id"],
    how="left",
)

# ---------------------------
# align column names to gatherer fold format
# ---------------------------
merged = merged.rename(columns={"text_id": "unique_paragraph_id"})

merged["unique_trial_id"] = (
    merged["participant_id"].astype(str)
    + "_"
    + merged["unique_paragraph_id"].astype(str)
    + "_1_0"
)

# desired output column order
target_cols = [
    "unique_paragraph_id",
    "participant_id",
    "unique_trial_id",
    "list_number",
    "question_preview",
    "repeated_reading_trial",
    "is_correct",
    "article_batch",
    "article_id",
    "regime",
]

# ---------------------------
# optional: quick diagnostics
# ---------------------------
print("Merged shape:", merged.shape)
print("\nMissing values per target column:")
print(merged[target_cols].isna().sum())


# ---------------------------
# cleanup: convert repeated_reading_trial to binary 0/1
# ---------------------------
merged["repeated_reading_trial"] = (
    merged["repeated_reading_trial"]
    .astype(str)
    .str.lower()
    .map({"true": 1, "false": 0})
)



merged["question_preview"] = merged["question_preview"].astype(str)

# ---------------------------
# write one file per fold
# ---------------------------
for fold_num in sorted(merged["fold"].dropna().unique()):
    fold_df = merged.loc[merged["fold"] == fold_num, target_cols].copy()

    out_path = output_dir / FOLD_TRIAL_IDS_FILENAME_TEMPLATE.format(fold_idx=int(fold_num))
    fold_df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}  ({len(fold_df)} rows)")

Merged shape: (97190, 13)

Missing values per target column:
unique_paragraph_id       0
participant_id            0
unique_trial_id           0
list_number               0
question_preview          0
repeated_reading_trial    0
is_correct                0
article_batch             0
article_id                0
regime                    0
dtype: int64
Saved: E:\QA PROJECT\programming\QA_eyetracking_workspace\data\HuntersFolds\fold_0_trial_ids_by_regime.csv  (9719 rows)
Saved: E:\QA PROJECT\programming\QA_eyetracking_workspace\data\HuntersFolds\fold_1_trial_ids_by_regime.csv  (9719 rows)
Saved: E:\QA PROJECT\programming\QA_eyetracking_workspace\data\HuntersFolds\fold_2_trial_ids_by_regime.csv  (9719 rows)
Saved: E:\QA PROJECT\programming\QA_eyetracking_workspace\data\HuntersFolds\fold_3_trial_ids_by_regime.csv  (9719 rows)
Saved: E:\QA PROJECT\programming\QA_eyetracking_workspace\data\HuntersFolds\fold_4_trial_ids_by_regime.csv  (9719 rows)
Saved: E:\QA PROJECT\programming\QA_eyetrackin